In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
model_llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)

In [4]:
class JokeState(TypedDict) :
    topic : str
    joke : str
    explanation : str

In [5]:
def gen_joke(state : JokeState) :
    prompt = f'Generate a joke on provided topic {state["topic"]}'

    response = model_llm.invoke(prompt).content

    return {'joke' : response}

In [6]:
def gen_explanation(state : JokeState) :
    prompt = f'Write an explanation for the joke - {state["joke"]}'

    response = model_llm.invoke(prompt).content

    return {'explanation' : response}

In [8]:
graph = StateGraph(JokeState)

graph.add_node('Joke', gen_joke)
graph.add_node('Explanation', gen_explanation)

graph.add_edge(START, 'Joke')
graph.add_edge('Joke', 'Explanation')
graph.add_edge('Explanation', END)

Checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=Checkpointer)

In [9]:
config1 = {'configurable' : {'thread_id' : "1"}}
workflow.invoke({'topic' : 'Pizza'}, config=config1)

{'topic': 'Pizza',
 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!',
 'explanation': 'A classic play on words. The joke relies on a clever pun to create humor. Here\'s how it works:\n\nThe setup "Why was the pizza in a bad mood?" primes the listener to expect a reason related to the pizza\'s emotional state. The punchline "Because it was feeling a little crusty!" subverts this expectation by using the word "crusty" in a double sense.\n\nIn one sense, "crusty" can describe someone or something that is irritable, gruff, or ill-tempered, which fits the context of the pizza being in a bad mood. However, "crusty" is also an adjective that literally describes the outer layer of a pizza, which is typically crispy and golden brown.\n\nThe joke exploits this dual meaning of "crusty" to create a wordplay that connects the pizza\'s emotional state to its physical characteristics. The humor comes from the unexpected twist on the word\'s meaning, creating a lig

In [10]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!', 'explanation': 'A classic play on words. The joke relies on a clever pun to create humor. Here\'s how it works:\n\nThe setup "Why was the pizza in a bad mood?" primes the listener to expect a reason related to the pizza\'s emotional state. The punchline "Because it was feeling a little crusty!" subverts this expectation by using the word "crusty" in a double sense.\n\nIn one sense, "crusty" can describe someone or something that is irritable, gruff, or ill-tempered, which fits the context of the pizza being in a bad mood. However, "crusty" is also an adjective that literally describes the outer layer of a pizza, which is typically crispy and golden brown.\n\nThe joke exploits this dual meaning of "crusty" to create a wordplay that connects the pizza\'s emotional state to its physical characteristics. The humor comes from the unexpected twist on the word\'s mean

In [12]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!', 'explanation': 'A classic play on words. The joke relies on a clever pun to create humor. Here\'s how it works:\n\nThe setup "Why was the pizza in a bad mood?" primes the listener to expect a reason related to the pizza\'s emotional state. The punchline "Because it was feeling a little crusty!" subverts this expectation by using the word "crusty" in a double sense.\n\nIn one sense, "crusty" can describe someone or something that is irritable, gruff, or ill-tempered, which fits the context of the pizza being in a bad mood. However, "crusty" is also an adjective that literally describes the outer layer of a pizza, which is typically crispy and golden brown.\n\nThe joke exploits this dual meaning of "crusty" to create a wordplay that connects the pizza\'s emotional state to its physical characteristics. The humor comes from the unexpected twist on the word\'s mea

In [14]:
config2 = {'configurable' : {'thread_id' : "2"}}
workflow.invoke({'topic' : 'Burger'}, config=config2)

{'topic': 'Burger',
 'joke': 'Why did the burger go to therapy?\n\nBecause it was feeling a little "well done" and needed to work through some beef with its emotions!',
 'explanation': 'A clever play on words. This joke relies on a pun to create humor. Here\'s a breakdown:\n\nThe setup "Why did the burger go to therapy?" primes the listener to expect a reason related to the burger\'s emotional state or mental health. The punchline "it was feeling a little \'well done\'" is a wordplay on the phrase "well done," which has a double meaning:\n\n1. **In cooking**: A burger can be cooked to various levels of doneness, with "well done" referring to a burger that is fully cooked and often dry.\n2. **In emotions**: "Well done" can also imply a sense of being overly anxious, stressed, or overwhelmed, similar to feeling "burned out."\n\nThe second part of the punchline, "and needed to work through some beef with its emotions," is another pun. "Beef" has a double meaning:\n\n1. **Food**: A burger 

In [15]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'Burger', 'joke': 'Why did the burger go to therapy?\n\nBecause it was feeling a little "well done" and needed to work through some beef with its emotions!', 'explanation': 'A clever play on words. This joke relies on a pun to create humor. Here\'s a breakdown:\n\nThe setup "Why did the burger go to therapy?" primes the listener to expect a reason related to the burger\'s emotional state or mental health. The punchline "it was feeling a little \'well done\'" is a wordplay on the phrase "well done," which has a double meaning:\n\n1. **In cooking**: A burger can be cooked to various levels of doneness, with "well done" referring to a burger that is fully cooked and often dry.\n2. **In emotions**: "Well done" can also imply a sense of being overly anxious, stressed, or overwhelmed, similar to feeling "burned out."\n\nThe second part of the punchline, "and needed to work through some beef with its emotions," is another pun. "Beef" has a double meaning:\n\n1. 